# Heat Equation Smoothing

Chapter 21: Fokker-Planck and Distribution Dynamics · *Mathematics for AI and Machine Learning* by Fuheng Wu

Forward diffusion broadens density and increases entropy over time.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-21/heat-smoothing/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(0)

# The demo's defaults: heat equation dp/dt = D p'' from p0 = mixture of N(-gap, 0.32^2) and N(gap, 0.32^2)
D, t, gap, s0 = 0.4, 1.2, 1.8, 0.32
x = torch.linspace(-6, 6, 4801, dtype=torch.float64)


def mixture(s):
    g = lambda m: torch.exp(-0.5 * ((x - m) / s) ** 2) / (math.sqrt(2 * math.pi) * s)
    return 0.5 * g(-gap) + 0.5 * g(gap)


def entropy(p):
    return -torch.trapezoid(p * torch.log(p.clamp_min(1e-300)), x).item()


# ── Heat flow = convolving with a Gaussian: each bump widens to sqrt(s0^2 + 2 D t)
st = math.sqrt(s0 ** 2 + 2 * D * t)
p0, pt = mixture(s0), mixture(st)
print(f"bump std: {s0} -> {st:.4f};  entropy {entropy(p0):.4f} -> {entropy(pt):.4f} (heat flow only increases it)")
print(f"density at the middle x = 0: {p0[len(x) // 2].item():.4f} -> {pt[len(x) // 2].item():.4f}")

# ── Check by solving the PDE numerically with finite differences ────────────
xs = x[::4]                                        # a coarser grid keeps the explicit scheme fast
dx = (xs[1] - xs[0]).item()
dt_num = 0.4 * dx ** 2 / D                         # explicit scheme needs dt <= dx^2 / (2 D)
p = p0[::4].clone()
for _ in range(round(t / dt_num)):
    lap = torch.zeros_like(p)
    lap[1:-1] = (p[2:] - 2 * p[1:-1] + p[:-2]) / dx ** 2
    p = p + D * dt_num * lap
print(f"finite-difference PDE vs closed form: max diff {(p - pt[::4]).abs().max().item():.2e}")

# ── Same thing with particles: x_t = x_0 + sqrt(2 D t) * noise ──────────────
n = 200000
x0 = torch.where(torch.rand(n) < 0.5, -gap, gap) + s0 * torch.randn(n)
xt = x0 + math.sqrt(2 * D * t) * torch.randn(n)
print(f"particles: std {xt.std().item():.4f}  vs  sqrt(gap^2 + st^2) = {math.sqrt(gap ** 2 + st ** 2):.4f}")

bump std: 0.32 -> 1.0307;  entropy 0.9727 -> 2.0379 (heat flow only increases it)
density at the middle x = 0: 0.0000 -> 0.0842
finite-difference PDE vs closed form: max diff 4.80e-05
particles: std 2.0745  vs  sqrt(gap^2 + st^2) = 2.0742
